In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sklearn.cluster import KMeans
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
from sksurv.datasets import load_gbsg2
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis, 
                            ExtraSurvivalTrees)
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
 
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
from sklearn.preprocessing import RobustScaler

In [2]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [3]:
# Need to choose patient_id from OUS_D1 in response_OUS
data = list(OUS_D1['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D2 with response_OUS
clinical_train = pd.merge(OUS_D1, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS', 'LRC', 'event_LRC'])]

In [4]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

In [5]:
# Check null values in D1
clinical_train.isnull().sum().sum()

0

## Test dataset: MAASTRO 

In [6]:
(MAASTRO_D1['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [7]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [8]:
# need to choose patient_id from MAASTRO_D1 in response_MAASTRO
data = list(MAASTRO_D1['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 

In [9]:
# Merge MAASTRO_D2 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D1, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'OS_event', 'LRC', 'LRC_event'])]

In [10]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [11]:
# Check if some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG,DFS,DFS_event


In [12]:
# X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS'])]

# y 
y = clinical_train.loc[:, ['DFS', 'event_DFS']]

In [13]:
# Set lower, upper time point and times for IBS calculation later 

# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

clinical_test.rename(columns = {'DFS_event' : 'event_DFS'}, inplace = True)

# X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'event_DFS'])]

# y y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['DFS', 'event_DFS']]
lower, upper = np.percentile(y_MAASTRO['DFS'], [10, 90])
times = np.arange(lower, upper)

# y into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_DFS'], y_MAASTRO['DFS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

X_train:  (139, 14)
y_train:  (139,)


(99, 16)

# Feature Selection: PLSR

In [14]:
selected_features = [
"MTV",
"TLG",
"hpv_related",
"SUVpeak",
"uicc8_III-IV",
"oropharynx"
]

# Selecting features in the DataFrame
X_plsr = X[selected_features]
X_new = X_plsr.copy()

In [15]:
X_MAASTRO_plsr = X_MAASTRO[selected_features]
MAASTRO_new = X_MAASTRO_plsr.copy()
# No need to standardize since all selected columns are binary
X_new_std = X_new
MAASTRO_new_std = MAASTRO_new

# Standardization

In [16]:
# Standardize X_new, the new data with the selected features only 
categorical_columns = ['female', 
                        'cavum_oris',
                        'oropharynx',
                        'hypopharynx',
                        'larynx',
                        'histgrade_high',
                        'hpv_related',
                        'charlson',
                        'uicc8_III-IV']

# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in X_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
X_new_categoric = X_new[columns_to_drop]
X_new_numeric = X_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
scaler = RobustScaler() 
X_new_numeric_columns = X_new_numeric.columns
X_new_numeric_index = X_new_numeric.index 

# All the columns are binary meaning no scaler needed 
X_new_numeric_std = scaler.fit_transform(X_new_numeric)
X_new_numeric_std = pd.DataFrame(X_new_numeric_std,
                                 columns=X_new_numeric_columns, 
                                 index=X_new_numeric_index)
X_new_std = pd.concat([X_new_numeric_std, X_new_categoric], axis=1)

# Change the order of the X_new_std 
X_new_std = X_new_std[X_new.columns]

In [17]:
# Standardize X_MAASTRO 
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in MAASTRO_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
MAASTRO_new_categoric = MAASTRO_new[columns_to_drop]
MAASTRO_new_numeric = MAASTRO_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 

MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_index = MAASTRO_new_numeric.index 
MAASTRO_new_numeric_std = scaler.transform(MAASTRO_new_numeric)
MAASTRO_new_numeric_std = pd.DataFrame(MAASTRO_new_numeric_std,
                                 columns=MAASTRO_new_numeric_columns, 
                                 index=MAASTRO_new_numeric_index)
MAASTRO_new_std = pd.concat([MAASTRO_new_numeric_std, MAASTRO_new_categoric], axis=1)

# Change the order of the MAASTRO_new, MAASTRO_new_std
MAASTRO_new = MAASTRO_new[X_new.columns]
MAASTRO_new_std = MAASTRO_new_std[X_new.columns]

In [18]:
X_new

,MTV,TLG,hpv_related,SUVpeak,uicc8_III-IV,oropharynx
0,7.934,86.228420,0.0,14.473272,0.0,1
1,1.656,7.040100,0.0,5.044678,0.0,0
2,14.502,83.569669,0.0,7.839043,1.0,0
3,2.440,5.567091,0.0,2.880631,0.0,0
4,3.668,16.150550,0.0,5.402006,0.0,0
...,...,...,...,...,...,...
134,3.650,26.280140,1.0,9.290139,0.0,1
135,18.967,101.754834,1.0,7.172883,1.0,1
136,6.370,66.273201,1.0,13.873187,0.0,1
137,12.443,71.832443,1.0,7.507419,1.0,1


In [19]:
X_new_std

,MTV,TLG,hpv_related,SUVpeak,uicc8_III-IV,oropharynx
0,0.075008,0.231096,0.0,0.600320,0.0,1
1,-0.466877,-0.387041,0.0,-0.690705,0.0,0
2,0.641923,0.210342,0.0,-0.308082,1.0,0
3,-0.399206,-0.398539,0.0,-0.987020,0.0,0
4,-0.293211,-0.315926,0.0,-0.641777,0.0,0
...,...,...,...,...,...,...
134,-0.294765,-0.236855,1.0,-0.109388,0.0,1
135,1.027319,0.352293,1.0,-0.399297,1.0,1
136,-0.059989,0.075327,1.0,0.518153,0.0,1
137,0.464201,0.118722,1.0,-0.353490,1.0,1


In [20]:
MAASTRO_new

,MTV,TLG,hpv_related,SUVpeak,uicc8_III-IV,oropharynx
0,22.841,263.611623,1,15.438583,0,1
1,5.660,36.980700,0,8.829353,1,1
2,7.791,74.636342,0,13.476123,1,1
3,7.908,46.791979,0,8.632732,1,0
4,15.237,107.637514,1,9.783954,0,1
...,...,...,...,...,...,...
94,6.110,144.490782,0,31.338410,1,0
95,7.182,69.214868,0,13.041604,1,0
96,16.483,102.594274,1,8.944517,1,1
97,9.981,103.229492,1,14.184236,0,1


In [21]:
MAASTRO_new_std

,MTV,TLG,hpv_related,SUVpeak,uicc8_III-IV,oropharynx
0,1.361702,1.615732,1,0.732497,0,1
1,-0.121272,-0.153327,0,-0.172482,1,1
2,0.062665,0.140609,0,0.463784,1,1
3,0.072763,-0.076742,0,-0.199405,1,0
4,0.705364,0.398213,1,-0.041772,0,1
...,...,...,...,...,...,...
94,-0.082431,0.685886,0,2.909606,1,0
95,0.010099,0.098289,0,0.404287,1,0
96,0.812913,0.358846,1,-0.156713,1,1
97,0.251694,0.363804,1,0.560744,0,1


# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [22]:
# Setting the y format for skf below  
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-18 14:51:54,768] A new study created in memory with name: no-name-dc9f2385-b4f3-4f07-888d-095b705329d2


  0%|          | 0/1 [00:00<?, ?it/s]

[I 2024-04-18 14:51:59,765] A new study created in memory with name: no-name-40498129-8a74-448e-9a84-f02803c23593


Fold 1 C-index: 0.5378486055776892
Fold 2 C-index: 0.6085271317829457
Fold 3 C-index: 0.7787234042553192
Fold 4 C-index: 0.6311787072243346
Fold 5 C-index: 0.6008583690987125
[I 2024-04-18 14:51:59,752] Trial 0 finished with value: 0.6314272435878002 and parameters: {}. Best is trial 0 with value: 0.6314272435878002.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6314272435878002], datetime_start=datetime.datetime(2024, 4, 18, 14, 51, 54, 856120), datetime_complete=datetime.datetime(2024, 4, 18, 14, 51, 59, 752382), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6314272435878002


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.25104063694870243
Fold 2 IBS: 0.23023130950396492
Fold 3 IBS: 0.1815270008194301
Fold 4 IBS: 0.28352164879300995
Fold 5 IBS: 0.2114703252261867
[I 2024-04-18 14:52:00,102] Trial 0 finished with value: 0.2315581842582588 and parameters: {}. Best is trial 0 with value: 0.2315581842582588.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.2315581842582588], datetime_start=datetime.datetime(2024, 4, 18, 14, 51, 59, 800110), datetime_complete=datetime.datetime(2024, 4, 18, 14, 52, 0, 102253), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.2315581842582588


In [23]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

In [24]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.631
train_ibs:  0.232


#### Test

In [25]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [26]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

CoxPHSurvivalAnalysis()

Concordance index: 0.593
IBS score: 0.253


In [27]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [28]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

### 2. CoxnetSurvivalAnalysis - Ridge 

#### Train

In [29]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-18 14:52:00,273] A new study created in memory with name: no-name-b1c411f0-5690-492c-b63a-9cc046bf8c21


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.5597609561752988
Fold 2 C-index: 0.6065891472868217
Fold 3 C-index: 0.6936170212765957


[I 2024-04-18 14:52:00,460] A new study created in memory with name: no-name-2ea39636-6c69-4965-b0f3-72b85290cdd4


Fold 4 C-index: 0.4695817490494297
Fold 5 C-index: 0.6180257510729614
[I 2024-04-18 14:52:00,452] Trial 0 finished with value: 0.5895149249722215 and parameters: {}. Best is trial 0 with value: 0.5895149249722215.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.5895149249722215], datetime_start=datetime.datetime(2024, 4, 18, 14, 52, 0, 311348), datetime_complete=datetime.datetime(2024, 4, 18, 14, 52, 0, 452738), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.5895149249722215


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.24724709622633062
Fold 2 IBS: 0.23203988074395682
Fold 3 IBS: 0.22898186249434238
Fold 4 IBS: 0.24197478945971077
Fold 5 IBS: 0.2293955885877772
[I 2024-04-18 14:52:00,682] Trial 0 finished with value: 0.23592784350242355 and parameters: {}. Best is trial 0 with value: 0.23592784350242355.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.23592784350242355], datetime_start=datetime.datetime(2024, 4, 18, 14, 52, 0, 497104), datetime_complete=datetime.datetime(2024, 4, 18, 14, 52, 0, 682266), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.23592784350242355


In [30]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [31]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.59
train_ibs:  0.236


#### Test

In [32]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [33]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.642


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.229


In [34]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [35]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-18 14:52:00,825] A new study created in memory with name: no-name-4ace507b-3bf0-4833-afe6-fde2e1e8202b


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.5378486055776892
Fold 2 C-index: 0.6124031007751938
Fold 3 C-index: 0.7659574468085106
Fold 4 C-index: 0.6349809885931559


[I 2024-04-18 14:52:01,160] A new study created in memory with name: no-name-08295041-18cb-4146-be8a-e08517f626b1


Fold 5 C-index: 0.6008583690987125
[I 2024-04-18 14:52:01,154] Trial 0 finished with value: 0.6304097021706524 and parameters: {}. Best is trial 0 with value: 0.6304097021706524.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6304097021706524], datetime_start=datetime.datetime(2024, 4, 18, 14, 52, 0, 858145), datetime_complete=datetime.datetime(2024, 4, 18, 14, 52, 1, 153853), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6304097021706524


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.2498479051521125
Fold 2 IBS: 0.22915617228216204
Fold 3 IBS: 0.18245597547256243
Fold 4 IBS: 0.2823066179106044
Fold 5 IBS: 0.21084019909088417
[I 2024-04-18 14:52:01,510] Trial 0 finished with value: 0.23092137398166512 and parameters: {}. Best is trial 0 with value: 0.23092137398166512.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.23092137398166512], datetime_start=datetime.datetime(2024, 4, 18, 14, 52, 1, 196933), datetime_complete=datetime.datetime(2024, 4, 18, 14, 52, 1, 510340), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.23092137398166512


In [36]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [37]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.63
train_ibs:  0.231


#### Test 

In [38]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [39]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.599


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.25


In [40]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [41]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-18 14:52:01,754] A new study created in memory with name: no-name-cbe9a5b9-15fb-469b-a63a-8dc09fe2d569


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5378486055776892
Fold 2 C-index: 0.6124031007751938
Fold 3 C-index: 0.7659574468085106
Fold 4 C-index: 0.6349809885931559
Fold 5 C-index: 0.6008583690987125
[I 2024-04-18 14:52:02,057] Trial 0 finished with value: 0.6304097021706524 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.6304097021706524.
Fold 1 C-index: 0.5378486055776892
Fold 2 C-index: 0.6124031007751938
Fold 3 C-index: 0.7617021276595745
Fold 4 C-index: 0.6311787072243346
Fold 5 C-index: 0.6008583690987125
[I 2024-04-18 14:52:02,364] Trial 1 finished with value: 0.6287981820671009 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.6304097021706524.
Fold 1 C-index: 0.5378486055776892
Fold 2 C-index: 0.6124031007751938
Fold 3 C-index: 0.7617021276595745
Fold 4 C-index: 0.6311787072243346
Fold 5 C-index: 0.6008583690987125
[I 2024-04-18 14:52:02,716] Trial 2 finished with value: 0.6287981820671009 and parameters: {'l1_ratio': 0.22692876841884668}.

Fold 2 C-index: 0.6124031007751938
Fold 3 C-index: 0.7702127659574468
Fold 4 C-index: 0.6349809885931559
Fold 5 C-index: 0.6008583690987125
[I 2024-04-18 14:52:08,654] Trial 24 finished with value: 0.6312607660004397 and parameters: {'l1_ratio': 0.6337782913213387}. Best is trial 17 with value: 0.6312607660004397.
Fold 1 C-index: 0.5378486055776892
Fold 2 C-index: 0.6124031007751938
Fold 3 C-index: 0.7617021276595745
Fold 4 C-index: 0.6349809885931559
Fold 5 C-index: 0.6008583690987125
[I 2024-04-18 14:52:08,911] Trial 25 finished with value: 0.6295586383408651 and parameters: {'l1_ratio': 0.4420371910071116}. Best is trial 17 with value: 0.6312607660004397.
Fold 1 C-index: 0.5378486055776892
Fold 2 C-index: 0.6124031007751938
Fold 3 C-index: 0.7659574468085106
Fold 4 C-index: 0.6349809885931559
Fold 5 C-index: 0.6008583690987125
[I 2024-04-18 14:52:09,158] Trial 26 finished with value: 0.6304097021706524 and parameters: {'l1_ratio': 0.9167426588025347}. Best is trial 17 with value: 0.

Fold 2 C-index: 0.6124031007751938
Fold 3 C-index: 0.7617021276595745
Fold 4 C-index: 0.6349809885931559
Fold 5 C-index: 0.6008583690987125
[I 2024-04-18 14:52:16,200] Trial 48 finished with value: 0.6295586383408651 and parameters: {'l1_ratio': 0.4037678712864638}. Best is trial 17 with value: 0.6312607660004397.
Fold 1 C-index: 0.5577689243027888
Fold 2 C-index: 0.622093023255814
Fold 3 C-index: 0.6936170212765957
Fold 4 C-index: 0.4714828897338403
Fold 5 C-index: 0.6266094420600858
[I 2024-04-18 14:52:16,344] Trial 49 finished with value: 0.5943142601258249 and parameters: {'l1_ratio': 0.005040986123852953}. Best is trial 17 with value: 0.6312607660004397.
Fold 1 C-index: 0.5378486055776892
Fold 2 C-index: 0.6124031007751938
Fold 3 C-index: 0.7659574468085106
Fold 4 C-index: 0.6349809885931559
Fold 5 C-index: 0.6008583690987125
[I 2024-04-18 14:52:16,645] Trial 50 finished with value: 0.6304097021706524 and parameters: {'l1_ratio': 0.705282490444848}. Best is trial 17 with value: 0.

...

  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 IBS: 0.2501164038101084
Fold 2 IBS: 0.22906885747870007
Fold 3 IBS: 0.18261608331177717
Fold 4 IBS: 0.28246778346363743
Fold 5 IBS: 0.21082694034208624
[I 2024-04-18 14:52:34,467] Trial 0 finished with value: 0.23101921368126183 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.23101921368126183.
Fold 1 IBS: 0.25045127911107684
Fold 2 IBS: 0.2289033481961742
Fold 3 IBS: 0.182812646795246
Fold 4 IBS: 0.28267568254733594
Fold 5 IBS: 0.2107988601018046
[I 2024-04-18 14:52:35,159] Trial 1 finished with value: 0.23112836335032752 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.23101921368126183.
Fold 1 IBS: 0.25053506079926924
Fold 2 IBS: 0.22896299234243309
Fold 3 IBS: 0.18274008272267475
Fold 4 IBS: 0.282689020442975
Fold 5 IBS: 0.21077984327525492
[I 2024-04-18 14:52:35,904] Trial 2 finished with value: 0.2311413999165214 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 0 with value: 0.23101921368126183

Fold 3 IBS: 0.18257521722964518
Fold 4 IBS: 0.28252916082691065
Fold 5 IBS: 0.21078283746483825
[I 2024-04-18 14:52:43,586] Trial 25 finished with value: 0.23101987000047292 and parameters: {'l1_ratio': 0.6427594368859761}. Best is trial 21 with value: 0.2309065397817624.
Fold 1 IBS: 0.24995997274999734
Fold 2 IBS: 0.22906065733451814
Fold 3 IBS: 0.1825100984425452
Fold 4 IBS: 0.2823823291034426
Fold 5 IBS: 0.21085394117755218
[I 2024-04-18 14:52:43,893] Trial 26 finished with value: 0.23095339976161106 and parameters: {'l1_ratio': 0.9167426588025347}. Best is trial 21 with value: 0.2309065397817624.
Fold 1 IBS: 0.24660182408601464
Fold 2 IBS: 0.2314408429286839
Fold 3 IBS: 0.22809923849026004
Fold 4 IBS: 0.2449941414355088
Fold 5 IBS: 0.22868896852966641
[I 2024-04-18 14:52:44,039] Trial 27 finished with value: 0.23596500309402674 and parameters: {'l1_ratio': 0.016560941611969082}. Best is trial 21 with value: 0.2309065397817624.
Fold 1 IBS: 0.24996010174254701
Fold 2 IBS: 0.229059770

Fold 3 IBS: 0.18269021411781583
Fold 4 IBS: 0.2825506909005969
Fold 5 IBS: 0.21082218123065888
[I 2024-04-18 14:52:50,839] Trial 50 finished with value: 0.23106296201521506 and parameters: {'l1_ratio': 0.556147470734502}. Best is trial 21 with value: 0.2309065397817624.
Fold 1 IBS: 0.24985451786514604
Fold 2 IBS: 0.22911757360111323
Fold 3 IBS: 0.18247745274466587
Fold 4 IBS: 0.28228934297583047
Fold 5 IBS: 0.210821944664204
[I 2024-04-18 14:52:51,195] Trial 51 finished with value: 0.23091216637019193 and parameters: {'l1_ratio': 0.9652982010146854}. Best is trial 21 with value: 0.2309065397817624.
Fold 1 IBS: 0.24985561903272943
Fold 2 IBS: 0.22911096435706593
Fold 3 IBS: 0.18248112989225465
Fold 4 IBS: 0.2822863805799371
Fold 5 IBS: 0.21081882479087846
[I 2024-04-18 14:52:51,506] Trial 52 finished with value: 0.2309105837305731 and parameters: {'l1_ratio': 0.9595865292004943}. Best is trial 21 with value: 0.2309065397817624.
Fold 1 IBS: 0.2499664669296027
Fold 2 IBS: 0.22912195392996

Fold 1 IBS: 0.24997685100061245
Fold 2 IBS: 0.22905113084183792
Fold 3 IBS: 0.18249549042198562
Fold 4 IBS: 0.2824457869682151
Fold 5 IBS: 0.21080036692776385
[I 2024-04-18 14:53:00,128] Trial 75 finished with value: 0.230953925232083 and parameters: {'l1_ratio': 0.8272338246472453}. Best is trial 63 with value: 0.23090537956938842.
Fold 1 IBS: 0.24996921705384784
Fold 2 IBS: 0.2291034328746273
Fold 3 IBS: 0.18250728530449697
Fold 4 IBS: 0.282355145135055
Fold 5 IBS: 0.21082539303666184
[I 2024-04-18 14:53:00,510] Trial 76 finished with value: 0.2309520946809378 and parameters: {'l1_ratio': 0.8672987568176258}. Best is trial 63 with value: 0.23090537956938842.
Fold 1 IBS: 0.24985859132527408
Fold 2 IBS: 0.22909284704859992
Fold 3 IBS: 0.18249120322574033
Fold 4 IBS: 0.2822782597115513
Fold 5 IBS: 0.2108102742239355
[I 2024-04-18 14:53:00,868] Trial 77 finished with value: 0.2309062351070202 and parameters: {'l1_ratio': 0.9442563293619028}. Best is trial 63 with value: 0.230905379569388

In [42]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [43]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.631
train_ibs:  0.231


#### Test

In [44]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [45]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.6422698927560148)

test_cindex : 0.599


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.9406994609406157)

test_ibs:  0.25


In [46]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [47]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-18 14:53:11,147] A new study created in memory with name: no-name-521d7482-4f7d-4cd3-ba32-d2f4cbc48b11


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5637450199203188
Fold 2 C-index: 0.6065891472868217
Fold 3 C-index: 0.7361702127659574
Fold 4 C-index: 0.6539923954372624
Fold 5 C-index: 0.5579399141630901
[I 2024-04-18 14:53:16,199] Trial 0 finished with value: 0.6236873379146901 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.6236873379146901.
Fold 1 C-index: 0.5936254980079682
Fold 2 C-index: 0.6356589147286822
Fold 3 C-index: 0.7063829787234043
Fold 4 C-index: 0.6463878326996197
Fold 5 C-index: 0.6137339055793991
[I 2024-04-18 14:53:19,803] Trial 1 finished with value: 0.6391578259478147 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717, 

Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-18 14:54:17,530] Trial 15 finished with value: 0.5 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 15, 'min_samples_leaf': 2, 'max_depth': 1, 'n_estimators': 385, 'oob_score': True, 'max_samples': 0.3692598016141903, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.18886845448542633, 'warm_start': True}. Best is trial 14 with value: 0.6653577067449185.
Fold 1 C-index: 0.5976095617529881
Fold 2 C-index: 0.7170542635658915
Fold 3 C-index: 0.8
Fold 4 C-index: 0.752851711026616
Fold 5 C-index: 0.6437768240343348
[I 2024-04-18 14:54:22,616] Trial 16 finished with value: 0.7022584720759661 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 15, 'min_samples_leaf': 1, 'max_depth': 7, 'n_estimators': 498, 'oob_score': True, 'max_samples': 0.47618376614423075, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.08910016992059805, 'warm_start': True}. Best is trial 16 with value: 0.7022584720759661.


Fold 1 C-index: 0.603585657370518
Fold 2 C-index: 0.6957364341085271
Fold 3 C-index: 0.7978723404255319
Fold 4 C-index: 0.7756653992395437
Fold 5 C-index: 0.6630901287553648
[I 2024-04-18 14:55:11,068] Trial 30 finished with value: 0.7071899919798972 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 18, 'min_samples_leaf': 15, 'max_depth': 16, 'n_estimators': 337, 'oob_score': True, 'max_samples': 0.9729345698205899, 'max_features': None, 'min_weight_fraction_leaf': 0.03900919066367596, 'warm_start': True}. Best is trial 30 with value: 0.7071899919798972.
Fold 1 C-index: 0.599601593625498
Fold 2 C-index: 0.687984496124031
Fold 3 C-index: 0.7978723404255319
Fold 4 C-index: 0.7756653992395437
Fold 5 C-index: 0.6630901287553648
[I 2024-04-18 14:55:14,122] Trial 31 finished with value: 0.7048427916339939 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 18, 'min_samples_leaf': 15, 'max_depth': 17, 'n_estimators': 345, 'oob_score': True, 'max_samples': 0.9530293221459376

...

Fold 1 IBS: 0.2487809856154952
Fold 2 IBS: 0.2357126863122274
Fold 3 IBS: 0.23121879278633423
Fold 4 IBS: 0.252716871302657
Fold 5 IBS: 0.2300529842210826
[I 2024-04-18 15:00:04,595] Trial 32 finished with value: 0.23969646404755926 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 16, 'min_samples_leaf': 8, 'max_depth': 15, 'n_estimators': 324, 'oob_score': False, 'max_samples': 0.684967894963397, 'max_features': None, 'min_weight_fraction_leaf': 0.33358217229374115}. Best is trial 22 with value: 0.2218187959851196.
Fold 1 IBS: 0.24085297329416872
Fold 2 IBS: 0.2345733046583843
Fold 3 IBS: 0.20759135963865943
Fold 4 IBS: 0.2365296995834454
Fold 5 IBS: 0.21299797485712868
[I 2024-04-18 15:00:10,611] Trial 33 finished with value: 0.22650906240635732 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 14, 'min_samples_leaf': 12, 'max_depth': 15, 'n_estimators': 377, 'oob_score': False, 'max_samples': 0.86361376148645, 'max_features': None, 'min_weight_fraction_leaf': 0.20

Fold 1 IBS: 0.2345170711209011
Fold 2 IBS: 0.2158702876822272
Fold 3 IBS: 0.20549905429174378
Fold 4 IBS: 0.2402979166180355
Fold 5 IBS: 0.21392455626227205
[I 2024-04-18 15:00:49,143] Trial 48 finished with value: 0.2220217771950359 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 19, 'min_samples_leaf': 11, 'max_depth': 19, 'n_estimators': 458, 'oob_score': False, 'max_samples': 0.9997930418100197, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.30064196242213814}. Best is trial 22 with value: 0.2218187959851196.
Fold 1 IBS: 0.24074271168077346
Fold 2 IBS: 0.22548053199748758
Fold 3 IBS: 0.2011822511428179
Fold 4 IBS: 0.23722082886916646
Fold 5 IBS: 0.21068234076308953
[I 2024-04-18 15:00:58,497] Trial 49 finished with value: 0.223061732890667 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 19, 'min_samples_leaf': 14, 'max_depth': 20, 'n_estimators': 463, 'oob_score': False, 'max_samples': 0.651099051191656, 'max_features': None, 'min_weight_fraction_leaf':

Fold 5 IBS: 0.21467305705178563
[I 2024-04-18 15:02:57,647] Trial 63 finished with value: 0.2218874083884687 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 20, 'min_samples_leaf': 19, 'max_depth': 17, 'n_estimators': 397, 'oob_score': True, 'max_samples': 0.9302168042817908, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.12589404086133016}. Best is trial 60 with value: 0.2216302684747876.
Fold 1 IBS: 0.23467329485216584
Fold 2 IBS: 0.2153250575930954
Fold 3 IBS: 0.20619628812969265
Fold 4 IBS: 0.23854415937650256
Fold 5 IBS: 0.21459253618191182
[I 2024-04-18 15:03:05,891] Trial 64 finished with value: 0.22186626722667366 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 20, 'min_samples_leaf': 19, 'max_depth': 17, 'n_estimators': 383, 'oob_score': True, 'max_samples': 0.9257891646726054, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.09737330304950191}. Best is trial 60 with value: 0.2216302684747876.
Fold 1 IBS: 0.23465626444537765
Fold 2 IBS: 0.21

Fold 1 IBS: 0.23840329527158535
Fold 2 IBS: 0.2240443733688464
Fold 3 IBS: 0.20985902553972755
Fold 4 IBS: 0.2414779454433706
Fold 5 IBS: 0.21993148705339322
[I 2024-04-18 15:05:19,646] Trial 79 finished with value: 0.22674322533538463 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 19, 'min_samples_leaf': 17, 'max_depth': 2, 'n_estimators': 500, 'oob_score': True, 'max_samples': 0.5136803098501763, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.044239866414212486}. Best is trial 77 with value: 0.2213480641065725.
Fold 1 IBS: 0.24167305181103554
Fold 2 IBS: 0.22408144386753678
Fold 3 IBS: 0.21862357706308283
Fold 4 IBS: 0.24304802780796642
Fold 5 IBS: 0.22308603875882582
[I 2024-04-18 15:05:27,856] Trial 80 finished with value: 0.2301024278616895 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 17, 'min_samples_leaf': 14, 'max_depth': 3, 'n_estimators': 464, 'oob_score': True, 'max_samples': 0.37703659520841803, 'max_features': 'sqrt', 'min_weight_fraction

Fold 5 IBS: 0.23029690187765103
[I 2024-04-18 15:07:10,743] Trial 94 finished with value: 0.23601966973656635 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 20, 'min_samples_leaf': 16, 'max_depth': 14, 'n_estimators': 307, 'oob_score': True, 'max_samples': 0.22295672704529856, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.06861072299069754}. Best is trial 77 with value: 0.2213480641065725.
Fold 1 IBS: 0.23464163108868474
Fold 2 IBS: 0.21657918908416698
Fold 3 IBS: 0.20594088914082598
Fold 4 IBS: 0.23877196303776577
Fold 5 IBS: 0.21486092621836406
[I 2024-04-18 15:07:17,700] Trial 95 finished with value: 0.2221589197139615 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 19, 'min_samples_leaf': 20, 'max_depth': 11, 'n_estimators': 408, 'oob_score': True, 'max_samples': 0.9110191728407264, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.1963462784654459}. Best is trial 77 with value: 0.2213480641065725.
Fold 1 IBS: 0.2351995544244154
Fold 2 IBS: 0.21

In [48]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [49]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.782
train_ibs:  0.221


#### Test

In [50]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))
    
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [51]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

RandomSurvivalForest(max_depth=20, max_leaf_nodes=8,
                     max_samples=0.891767758010405, min_samples_leaf=1,
                     min_samples_split=5,
                     min_weight_fraction_leaf=0.01360705866922349,
                     n_estimators=306, random_state=123, warm_start=True)

test_cindex:  0.622


RandomSurvivalForest(max_depth=1, max_leaf_nodes=20,
                     max_samples=0.9412925565225843, min_samples_leaf=18,
                     min_samples_split=14,
                     min_weight_fraction_leaf=0.1648569090893628,
                     n_estimators=483, oob_score=True, random_state=123)

test_ibs:  0.214


In [52]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [53]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-18 15:08:07,579] A new study created in memory with name: no-name-f2350a36-c2db-4dd1-9ac2-816c844605ae


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6135458167330677
Fold 2 C-index: 0.686046511627907
Fold 3 C-index: 0.7574468085106383
Fold 4 C-index: 0.7167300380228137
Fold 5 C-index: 0.592274678111588
[I 2024-04-18 15:08:08,939] Trial 0 finished with value: 0.6732087706012029 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7641958651588321, 'min_weight_fraction_leaf': 0.09124586522674999}. Best is trial 0 with value: 0.6732087706012029.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-18 15:08:11,781] Trial 1 finished with value: 0.5 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 425, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.4877764869966794, 'min_weight_fraction_leaf': 0.2468425488251531}.

Fold 1 C-index: 0.5617529880478087
Fold 2 C-index: 0.6298449612403101
Fold 3 C-index: 0.7553191489361702
Fold 4 C-index: 0.6806083650190115
Fold 5 C-index: 0.6008583690987125
[I 2024-04-18 15:08:45,997] Trial 16 finished with value: 0.6456767664684027 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 17, 'min_samples_leaf': 5, 'max_depth': 20, 'n_estimators': 499, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.8750346354626457, 'min_weight_fraction_leaf': 0.4093818399278544}. Best is trial 12 with value: 0.6812566420569242.
Fold 1 C-index: 0.5756972111553785
Fold 2 C-index: 0.6492248062015504
Fold 3 C-index: 0.7680851063829788
Fold 4 C-index: 0.7110266159695817
Fold 5 C-index: 0.628755364806867
[I 2024-04-18 15:08:47,286] Trial 17 finished with value: 0.6665578209032713 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 10, 'min_samples_leaf': 8, 'max_depth': 12, 'n_estimators': 318, 'oob_score': False, 'warm_start': True, 'max_featu

Fold 1 C-index: 0.6095617529880478
Fold 2 C-index: 0.689922480620155
Fold 3 C-index: 0.7404255319148936
Fold 4 C-index: 0.7319391634980988
Fold 5 C-index: 0.6866952789699571
[I 2024-04-18 15:09:12,923] Trial 31 finished with value: 0.6917088415982304 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 5, 'min_samples_leaf': 2, 'max_depth': 9, 'n_estimators': 281, 'oob_score': False, 'warm_start': True, 'max_features': None, 'max_samples': 0.6298966865274057, 'min_weight_fraction_leaf': 0.05405927482118079}. Best is trial 23 with value: 0.6975178627058407.
Fold 1 C-index: 0.6135458167330677
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.7531914893617021
Fold 4 C-index: 0.7338403041825095
Fold 5 C-index: 0.6609442060085837
[I 2024-04-18 15:09:14,079] Trial 32 finished with value: 0.6918392469781028 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 2, 'max_depth': 8, 'n_estimators': 289, 'oob_score': False, 'warm_start': True, 'max_features': N

...

Fold 1 IBS: 0.23788847993111656
Fold 2 IBS: 0.2180765411385516
Fold 3 IBS: 0.2046960039221167
Fold 4 IBS: 0.2247765505967664
Fold 5 IBS: 0.2077685664696797
[I 2024-04-18 15:13:36,700] Trial 30 finished with value: 0.2186412284116462 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 18, 'min_samples_leaf': 16, 'max_depth': 12, 'n_estimators': 343, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.889360966245883, 'min_weight_fraction_leaf': 0.14458594432203747}. Best is trial 11 with value: 0.21831633508886342.
Fold 1 IBS: 0.23712385212056705
Fold 2 IBS: 0.21925458444400572
Fold 3 IBS: 0.20321552597569
Fold 4 IBS: 0.2255422658404121
Fold 5 IBS: 0.20908692558988173
[I 2024-04-18 15:13:41,605] Trial 31 finished with value: 0.21884463079411134 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 10, 'min_samples_leaf': 20, 'max_depth': 17, 'n_estimators': 379, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7188

Fold 1 IBS: 0.23747885558720908
Fold 2 IBS: 0.2182158397444273
Fold 3 IBS: 0.2051184817191784
Fold 4 IBS: 0.22477313569503773
Fold 5 IBS: 0.20779683491497394
[I 2024-04-18 15:14:49,271] Trial 45 finished with value: 0.2186766295321653 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 3, 'min_samples_leaf': 5, 'max_depth': 19, 'n_estimators': 423, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.8732282072435447, 'min_weight_fraction_leaf': 0.21737715090273346}. Best is trial 11 with value: 0.21831633508886342.
Fold 1 IBS: 0.24146274510328755
Fold 2 IBS: 0.22120633219111893
Fold 3 IBS: 0.20427581901257044
Fold 4 IBS: 0.2260424188972459
Fold 5 IBS: 0.20767896038910674
[I 2024-04-18 15:14:53,994] Trial 46 finished with value: 0.22013325511866594 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 5, 'min_samples_leaf': 2, 'max_depth': 18, 'n_estimators': 436, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.53

Fold 1 IBS: 0.2377052136548948
Fold 2 IBS: 0.21605965008867054
Fold 3 IBS: 0.20558825307293532
Fold 4 IBS: 0.2254323656319241
Fold 5 IBS: 0.2067921952642682
[I 2024-04-18 15:15:48,540] Trial 60 finished with value: 0.21831553554253857 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 11, 'min_samples_leaf': 6, 'max_depth': 5, 'n_estimators': 102, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.2992752088637204, 'min_weight_fraction_leaf': 0.05635944811975884}. Best is trial 60 with value: 0.21831553554253857.
Fold 1 IBS: 0.23794023360294106
Fold 2 IBS: 0.2171601602160643
Fold 3 IBS: 0.2061252197645397
Fold 4 IBS: 0.22518242245675577
Fold 5 IBS: 0.20728448682330153
[I 2024-04-18 15:15:49,758] Trial 61 finished with value: 0.21873850457272045 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 11, 'min_samples_leaf': 6, 'max_depth': 4, 'n_estimators': 97, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.3418

Fold 1 IBS: 0.2380315443396539
Fold 2 IBS: 0.218584818434336
Fold 3 IBS: 0.2090327196956189
Fold 4 IBS: 0.22815174669205668
Fold 5 IBS: 0.2127482081129169
[I 2024-04-18 15:16:33,530] Trial 75 finished with value: 0.22130980745491646 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 8, 'min_samples_leaf': 6, 'max_depth': 19, 'n_estimators': 420, 'oob_score': False, 'warm_start': True, 'max_features': 1, 'max_samples': 0.6496285186532654, 'min_weight_fraction_leaf': 0.09733418927981143}. Best is trial 67 with value: 0.2182330563172799.
Fold 1 IBS: 0.23951407371043723
Fold 2 IBS: 0.2236154605317524
Fold 3 IBS: 0.21733512904776037
Fold 4 IBS: 0.23282386265018934
Fold 5 IBS: 0.21729889644220104
[I 2024-04-18 15:16:36,190] Trial 76 finished with value: 0.2261174844764681 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 11, 'min_samples_leaf': 19, 'max_depth': 16, 'n_estimators': 266, 'oob_score': False, 'warm_start': True, 'max_features': 0.1, 'max_samples': 0.575684434356

Fold 1 IBS: 0.24306418653288445
Fold 2 IBS: 0.22515153493924325
Fold 3 IBS: 0.1975215183442548
Fold 4 IBS: 0.22885063171920877
Fold 5 IBS: 0.20614416280161527
[I 2024-04-18 15:17:37,752] Trial 90 finished with value: 0.2201464068674413 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 15, 'min_samples_leaf': 4, 'max_depth': 20, 'n_estimators': 456, 'oob_score': False, 'warm_start': True, 'max_features': None, 'max_samples': 0.7361210166358332, 'min_weight_fraction_leaf': 0.2524918082691855}. Best is trial 67 with value: 0.2182330563172799.
Fold 1 IBS: 0.23776347268963513
Fold 2 IBS: 0.21793810774310016
Fold 3 IBS: 0.2045185088977267
Fold 4 IBS: 0.22480630923293607
Fold 5 IBS: 0.20794858004520358
[I 2024-04-18 15:17:42,376] Trial 91 finished with value: 0.2185949957217203 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 9, 'min_samples_leaf': 3, 'max_depth': 15, 'n_estimators': 391, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7668

In [54]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [55]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.746
train_ibs:  0.218


#### Test

In [56]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [57]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ExtraSurvivalTrees(max_depth=6, max_features=None, max_leaf_nodes=8,
                   max_samples=0.949451910660598, min_samples_leaf=4,
                   min_samples_split=15,
                   min_weight_fraction_leaf=0.030419386073469834,
                   n_estimators=377, random_state=123, warm_start=True)

C-index score: 0.6


ExtraSurvivalTrees(max_depth=8, max_features='log2', max_leaf_nodes=9,
                   max_samples=0.7501323059528776, min_samples_leaf=4,
                   min_weight_fraction_leaf=0.21276472095633941,
                   n_estimators=322, random_state=123, warm_start=True)

IBS: 0.213


In [58]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis

#### Train

In [59]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-18 15:18:24,809] A new study created in memory with name: no-name-d8d0d603-a840-43b7-8f40-738dc1e833dc


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-18 15:18:50,874] Trial 0 finished with value: 0.5 and parameters: {'subsample': 0.7268222670380755, 'learning_rate': 0.02932779416008757, 'dropout_rate': 0.3041663082077828, 'n_estimators': 276, 'criterion': 'friedman_mse', 'ccp_alpha': 9.807641983846155, 'min_weight_fraction_leaf': 0.34241486929243165, 'max_features': None, 'min_impurity_decrease': 2.4449249473284515e-05, 'validation_fraction': 0.7379954057320357, 'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 11, 'max_depth': 11}. Best is trial 0 with value: 0.5.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-18 15:19:17,827] Trial 1 finished with value: 0.5 and parameters: {'subsample': 0.6709608626961889, 'learning_rate': 0.08509374761370117, 'dropout_rate': 0.7520097923745717, 'n_estimators': 306, 'criterion': 'friedman_mse', 'ccp_alpha': 3.

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-18 15:26:48,893] Trial 13 finished with value: 0.5 and parameters: {'subsample': 0.8386796524426539, 'learning_rate': 0.046734492485875676, 'dropout_rate': 0.4821375662037144, 'n_estimators': 402, 'criterion': 'squared_error', 'ccp_alpha': 1.5696007313501796, 'min_weight_fraction_leaf': 0.18684147934268416, 'max_features': 'log2', 'min_impurity_decrease': 1.5044881127471587e-06, 'validation_fraction': 0.8166356053932342, 'min_samples_split': 16, 'max_leaf_nodes': 19, 'min_samples_leaf': 16, 'max_depth': 4}. Best is trial 9 with value: 0.6257330792225687.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-18 15:27:36,095] Trial 14 finished with value: 0.5 and parameters: {'subsample': 0.33235389014851724, 'learning_rate': 0.04522573411670834, 'dropout_rate': 0.2712811374536856, 'n_estimators': 405, 'criterion': 'square

Fold 5 C-index: 0.5
[I 2024-04-18 15:40:00,241] Trial 25 finished with value: 0.5 and parameters: {'subsample': 0.7565257046780437, 'learning_rate': 0.01188344684416992, 'dropout_rate': 0.3433523077110169, 'n_estimators': 318, 'criterion': 'squared_error', 'ccp_alpha': 2.063559212730723, 'min_weight_fraction_leaf': 0.25401273903421573, 'max_features': 'auto', 'min_impurity_decrease': 5.773435946665558e-07, 'validation_fraction': 0.8062268184400477, 'min_samples_split': 16, 'max_leaf_nodes': 18, 'min_samples_leaf': 5, 'max_depth': 3}. Best is trial 22 with value: 0.6567802711331774.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-18 15:41:08,144] Trial 26 finished with value: 0.5 and parameters: {'subsample': 0.8922404482621683, 'learning_rate': 0.011585260292674536, 'dropout_rate': 0.2527000999648632, 'n_estimators': 446, 'criterion': 'friedman_mse', 'ccp_alpha': 0.11820935501930148, 'min_weight_fraction_leaf': 0.3509209656

...

Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-18 20:50:05,207] Trial 55 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.804828042321909, 'learning_rate': 0.023646082998228058, 'dropout_rate': 0.23926982708415356, 'n_estimators': 162, 'criterion': 'squared_error', 'ccp_alpha': 0.40074886285106287, 'min_weight_fraction_leaf': 0.2779066644542128, 'max_features': 'sqrt', 'min_impurity_decrease': 0.008146563872249941, 'validation_fraction': 0.8868940629916056, 'min_samples_split': 4, 'max_leaf_nodes': 16, 'min_samples_leaf': 19, 'max_depth': 4}. Best is trial 45 with value: 0.23368011393511398.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-18 20:50:59,352] Trial 56 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.6106355356441384, 'learning_rate': 0.017538150300

Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-18 20:55:51,521] Trial 66 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9043457557527063, 'learning_rate': 0.08720647345857328, 'dropout_rate': 0.27193939046019544, 'n_estimators': 154, 'criterion': 'squared_error', 'ccp_alpha': 1.4111498627316026, 'min_weight_fraction_leaf': 0.23517339076530247, 'max_features': 'sqrt', 'min_impurity_decrease': 6.086951842225704e-07, 'validation_fraction': 0.9127218528145804, 'min_samples_split': 4, 'max_leaf_nodes': 15, 'min_samples_leaf': 20, 'max_depth': 2}. Best is trial 63 with value: 0.23182692024200646.
Fold 1 IBS: 0.24724710044658996
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-18 20:55:57,709] Trial 67 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.8250695857763196, 'learning_rate': 0.09110863428

Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-18 21:02:51,624] Trial 77 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.7428517276581355, 'learning_rate': 0.07706989744762442, 'dropout_rate': 0.4476209058480567, 'n_estimators': 91, 'criterion': 'squared_error', 'ccp_alpha': 1.5888723787914292, 'min_weight_fraction_leaf': 0.16206442611838873, 'max_features': 0.1, 'min_impurity_decrease': 3.33671674086432e-07, 'validation_fraction': 0.9402673106586569, 'min_samples_split': 12, 'max_leaf_nodes': 13, 'min_samples_leaf': 17, 'max_depth': 9}. Best is trial 63 with value: 0.23182692024200646.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-18 21:03:22,590] Trial 78 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.7715268987373948, 'learning_rate': 0.0898550375492334

Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927118
Fold 5 IBS: 0.2293955930480925
[I 2024-04-18 21:06:45,768] Trial 88 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.6413430388178152, 'learning_rate': 0.0973528090376961, 'dropout_rate': 0.26912242502602596, 'n_estimators': 174, 'criterion': 'squared_error', 'ccp_alpha': 0.5807061908097002, 'min_weight_fraction_leaf': 0.16723537370276506, 'max_features': 0.1, 'min_impurity_decrease': 1.190634063232502e-06, 'validation_fraction': 0.8005899065401637, 'min_samples_split': 9, 'max_leaf_nodes': 11, 'min_samples_leaf': 17, 'max_depth': 4}. Best is trial 82 with value: 0.23162915913259985.
Fold 1 IBS: 0.2433992658820838
Fold 2 IBS: 0.2267996492121836
Fold 3 IBS: 0.22140115508973934
Fold 4 IBS: 0.23912914067521288
Fold 5 IBS: 0.22390565225002362
[I 2024-04-18 21:07:22,324] Trial 89 finished with value: 0.23092697262184864 and parameters: {'subsample': 0.38730242337646104, 'learning_rate': 0.081098008363879

Fold 4 IBS: 0.23930587351662694
Fold 5 IBS: 0.22629676970219087
[I 2024-04-18 21:14:12,932] Trial 99 finished with value: 0.23188101200067718 and parameters: {'subsample': 0.40791967441342253, 'learning_rate': 0.07712333235501795, 'dropout_rate': 0.19859974199600133, 'n_estimators': 143, 'criterion': 'squared_error', 'ccp_alpha': 0.004426170739568851, 'min_weight_fraction_leaf': 0.16314282359709542, 'max_features': 'log2', 'min_impurity_decrease': 5.353566353972388e-07, 'validation_fraction': 0.1604355124738367, 'min_samples_split': 11, 'max_leaf_nodes': 15, 'min_samples_leaf': 14, 'max_depth': 4}. Best is trial 91 with value: 0.2298103711566252.


* Best trial for IBS: 
 FrozenTrial(number=91, state=TrialState.COMPLETE, values=[0.2298103711566252], datetime_start=datetime.datetime(2024, 4, 18, 21, 8, 36, 933618), datetime_complete=datetime.datetime(2024, 4, 18, 21, 9, 14, 185559), params={'subsample': 0.3646963250854607, 'learning_rate': 0.09255241886126767, 'dropout_rate': 0.12363276

In [60]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [61]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.663
train_ibs:  0.23


#### Test

In [62]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [63]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(ccp_alpha=0.009781579144263263,
                                 criterion='squared_error',
                                 dropout_rate=0.2158120114915196,
                                 learning_rate=0.05877903849297955,
                                 max_depth=20, max_features='sqrt',
                                 max_leaf_nodes=16,
                                 min_impurity_decrease=5.215208696037137e-07,
                                 min_samples_leaf=17, min_samples_split=17,
                                 min_weight_fraction_leaf=0.33111548353893067,
                                 n_estimators=28, random_state=123,
                                 subsample=0.7305170806182099,
                                 validation_fraction=0.8384582646351804)

C-index score: 0.622


GradientBoostingSurvivalAnalysis(ccp_alpha=0.03832008885345805,
                                 criterion='squared_error',
                                 dropout_rate=0.12363276406693596,
                                 learning_rate=0.09255241886126767, max_depth=4,
                                 max_leaf_nodes=15,
                                 min_impurity_decrease=6.12703415921239e-07,
                                 min_samples_leaf=16, min_samples_split=13,
                                 min_weight_fraction_leaf=0.14198218649980035,
                                 n_estimators=143, random_state=123,
                                 subsample=0.3646963250854607,
                                 validation_fraction=0.965806971606102)

IBS: 0.221


In [64]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

#### Train

In [65]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-18 21:14:21,150] A new study created in memory with name: no-name-a7d749c1-0c92-42f4-b129-f62146ad73ee


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5976095617529881
Fold 2 C-index: 0.6104651162790697
Fold 3 C-index: 0.7446808510638298
Fold 4 C-index: 0.5855513307984791
Fold 5 C-index: 0.6244635193133047
[I 2024-04-18 21:14:25,058] Trial 0 finished with value: 0.6325540758415343 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.6325540758415343.
Fold 1 C-index: 0.5358565737051793
Fold 2 C-index: 0.6104651162790697
Fold 3 C-index: 0.7446808510638298
Fold 4 C-index: 0.5855513307984791
Fold 5 C-index: 0.6244635193133047
[I 2024-04-18 21:14:59,499] Trial 1 finished with value: 0.6202034782319725 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 0 with value: 0.6325540758415343.
Fold 1 C-index: 0.5976095617529881
Fold 2 C-index: 0.6104651162790697
Fold 3 C-index: 0.7106382978723405
Fold 

Fold 1 C-index: 0.5358565737051793
Fold 2 C-index: 0.6104651162790697
Fold 3 C-index: 0.6787234042553192
Fold 4 C-index: 0.5855513307984791
Fold 5 C-index: 0.6244635193133047
[I 2024-04-18 21:45:31,022] Trial 19 finished with value: 0.6070119888702704 and parameters: {'subsample': 0.8350722272141606, 'dropout_rate': 0.5915597174096103, 'n_estimators': 299, 'learning_rate': 0.08697733590757546}. Best is trial 0 with value: 0.6325540758415343.
Fold 1 C-index: 0.5697211155378487
Fold 2 C-index: 0.6104651162790697
Fold 3 C-index: 0.6978723404255319
Fold 4 C-index: 0.5855513307984791
Fold 5 C-index: 0.6244635193133047
[I 2024-04-18 21:46:00,654] Trial 20 finished with value: 0.6176146844708469 and parameters: {'subsample': 0.38161734041406226, 'dropout_rate': 0.7859448591953863, 'n_estimators': 408, 'learning_rate': 0.06348170552170027}. Best is trial 0 with value: 0.6325540758415343.
Fold 1 C-index: 0.5358565737051793
Fold 2 C-index: 0.6104651162790697
Fold 3 C-index: 0.7446808510638298
Fo

Fold 1 C-index: 0.5358565737051793
Fold 2 C-index: 0.6104651162790697
Fold 3 C-index: 0.6787234042553192
Fold 4 C-index: 0.5855513307984791
Fold 5 C-index: 0.6244635193133047
[I 2024-04-18 21:50:19,203] Trial 38 finished with value: 0.6070119888702704 and parameters: {'subsample': 0.8975124960092108, 'dropout_rate': 0.3454513035831812, 'n_estimators': 238, 'learning_rate': 0.050384430919465664}. Best is trial 0 with value: 0.6325540758415343.
Fold 1 C-index: 0.5976095617529881
Fold 2 C-index: 0.6104651162790697
Fold 3 C-index: 0.7446808510638298
Fold 4 C-index: 0.5855513307984791
Fold 5 C-index: 0.6244635193133047
[I 2024-04-18 21:50:35,153] Trial 39 finished with value: 0.6325540758415343 and parameters: {'subsample': 0.7415710071500535, 'dropout_rate': 0.8737909302085629, 'n_estimators': 357, 'learning_rate': 0.01715036845928685}. Best is trial 0 with value: 0.6325540758415343.
Fold 1 C-index: 0.5976095617529881
Fold 2 C-index: 0.6104651162790697
Fold 3 C-index: 0.7446808510638298
Fo

...

Fold 2 IBS: 0.22156847431530372
Fold 3 IBS: 0.2084967358185174
Fold 4 IBS: 0.26598308545567084
Fold 5 IBS: 0.21358819750439922
[I 2024-04-18 22:04:23,324] Trial 19 finished with value: 0.2296383534267191 and parameters: {'subsample': 0.10013306718236009, 'dropout_rate': 0.9930703343219778, 'n_estimators': 51, 'learning_rate': 0.04096883532946153}. Best is trial 14 with value: 0.22913043364981825.
Fold 1 IBS: 0.24663727529252127
Fold 2 IBS: 0.21835663565155092
Fold 3 IBS: 0.2105256665576795
Fold 4 IBS: 0.24932331942924524
Fold 5 IBS: 0.21172241066007214
[I 2024-04-18 22:04:28,721] Trial 20 finished with value: 0.22731306151821382 and parameters: {'subsample': 0.8452240769714876, 'dropout_rate': 0.5558690358940764, 'n_estimators': 145, 'learning_rate': 0.017331004877487094}. Best is trial 20 with value: 0.22731306151821382.
Fold 1 IBS: 0.24576238490185268
Fold 2 IBS: 0.2188752760756841
Fold 3 IBS: 0.2117843863259187
Fold 4 IBS: 0.24706679820559785
Fold 5 IBS: 0.21313004630122903
[I 2024-

Fold 2 IBS: 0.2182215593937306
Fold 3 IBS: 0.20823493346801503
Fold 4 IBS: 0.2618867894566613
Fold 5 IBS: 0.2093390154585288
[I 2024-04-18 22:06:34,754] Trial 38 finished with value: 0.22936298071305022 and parameters: {'subsample': 0.7636971218212085, 'dropout_rate': 0.6483709169497328, 'n_estimators': 157, 'learning_rate': 0.019144317545493773}. Best is trial 23 with value: 0.2269899127562185.
Fold 1 IBS: 0.28103916288812086
Fold 2 IBS: 0.22171438039892677
Fold 3 IBS: 0.20500749397585746
Fold 4 IBS: 0.27630173359496024
Fold 5 IBS: 0.20128638585550865
[I 2024-04-18 22:06:46,677] Trial 39 finished with value: 0.2370698313426748 and parameters: {'subsample': 0.9093398198310482, 'dropout_rate': 0.35016062952952165, 'n_estimators': 241, 'learning_rate': 0.027420380756577795}. Best is trial 23 with value: 0.2269899127562185.
Fold 1 IBS: 0.33710682228256955
Fold 2 IBS: 0.3001823443008312
Fold 3 IBS: 0.24659723127721248
Fold 4 IBS: 0.3977251462922417
Fold 5 IBS: 0.2346436471139825
[I 2024-04

Fold 2 IBS: 0.2234966644079048
Fold 3 IBS: 0.20593980844129078
Fold 4 IBS: 0.27519054629155376
Fold 5 IBS: 0.20167917402674448
[I 2024-04-18 22:09:19,335] Trial 57 finished with value: 0.23862220409358526 and parameters: {'subsample': 0.9320999261183475, 'dropout_rate': 0.4172035801671631, 'n_estimators': 196, 'learning_rate': 0.036525597651522655}. Best is trial 56 with value: 0.22540990549430515.
Fold 1 IBS: 0.25001648359878603
Fold 2 IBS: 0.21622148540985112
Fold 3 IBS: 0.20811978633565079
Fold 4 IBS: 0.23691774104033314
Fold 5 IBS: 0.2092644296246439
[I 2024-04-18 22:09:29,395] Trial 58 finished with value: 0.224107985201853 and parameters: {'subsample': 0.9586197943864213, 'dropout_rate': 0.5740168344366231, 'n_estimators': 232, 'learning_rate': 0.013408218059981283}. Best is trial 58 with value: 0.224107985201853.
Fold 1 IBS: 0.2516874301562455
Fold 2 IBS: 0.21553073521939023
Fold 3 IBS: 0.20726808939440847
Fold 4 IBS: 0.23795443014521295
Fold 5 IBS: 0.20905124507220235
[I 2024-0

Fold 3 IBS: 0.2055220508619335
Fold 4 IBS: 0.2400446483858902
Fold 5 IBS: 0.20904869748033597
[I 2024-04-18 22:13:52,687] Trial 76 finished with value: 0.22528262013167547 and parameters: {'subsample': 0.9850530223586534, 'dropout_rate': 0.8436461076063504, 'n_estimators': 315, 'learning_rate': 0.01272054974228212}. Best is trial 58 with value: 0.224107985201853.
Fold 1 IBS: 0.2483081177028861
Fold 2 IBS: 0.21709564903157766
Fold 3 IBS: 0.20941684673788108
Fold 4 IBS: 0.242767563788497
Fold 5 IBS: 0.21032040763749132
[I 2024-04-18 22:14:10,793] Trial 77 finished with value: 0.22558171697966664 and parameters: {'subsample': 0.9014722175572065, 'dropout_rate': 0.9195364994175442, 'n_estimators': 394, 'learning_rate': 0.007105228355913655}. Best is trial 58 with value: 0.224107985201853.
Fold 1 IBS: 0.2866109326332082
Fold 2 IBS: 0.2278685488228803
Fold 3 IBS: 0.200404072315968
Fold 4 IBS: 0.2933783890804398
Fold 5 IBS: 0.20174961400213423
[I 2024-04-18 22:14:33,823] Trial 78 finished wit

Fold 4 IBS: 0.2383428523814535
Fold 5 IBS: 0.22304882189394717
[I 2024-04-18 23:30:21,650] Trial 95 finished with value: 0.23080795146039007 and parameters: {'subsample': 0.9761469148364865, 'dropout_rate': 0.9976109764514575, 'n_estimators': 310, 'learning_rate': 0.0024676593595529574}. Best is trial 58 with value: 0.224107985201853.
Fold 1 IBS: 0.26782485208938583
Fold 2 IBS: 0.21701225468661342
Fold 3 IBS: 0.20414842345737763
Fold 4 IBS: 0.25182349910467233
Fold 5 IBS: 0.20356211708550198
[I 2024-04-18 23:30:40,009] Trial 96 finished with value: 0.2288742292847102 and parameters: {'subsample': 0.932574881679507, 'dropout_rate': 0.8491419147294211, 'n_estimators': 363, 'learning_rate': 0.014382672512565652}. Best is trial 58 with value: 0.224107985201853.
Fold 1 IBS: 0.24422362832471908
Fold 2 IBS: 0.2208135515925996
Fold 3 IBS: 0.2160339018757797
Fold 4 IBS: 0.23597229499128605
Fold 5 IBS: 0.21768932618988893
[I 2024-04-18 23:30:50,260] Trial 97 finished with value: 0.22694654059485

In [66]:
train_cindex['ComponentwiseGradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['ComponentwiseGradientBoosting'] = np.round(study_ibs.best_value, 3)

In [67]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.633
train_ibs:  0.224


#### Test

In [68]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [69]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.3575254014553415,
                                              learning_rate=0.05558016213920623,
                                              n_estimators=114,
                                              random_state=123,
                                              subsample=0.7268222670380755)

C-index score: 0.601


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.5740168344366231,
                                              learning_rate=0.013408218059981283,
                                              n_estimators=232,
                                              random_state=123,
                                              subsample=0.9586197943864213)

IBS: 0.214


In [70]:
# Saving the values to the dictionary 
test_cindex['ComponentwiseGradientBoosting'] = c_index
test_ibs['ComponentwiseGradientBoosting'] = ibs

## Results

In [71]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

,C-index,rank
Randomsurvivalforest,0.782,1.0
ExtraSurvivalTrees,0.746,2.0
GradientBoosting,0.663,3.0
ComponentwiseGradientBoosting,0.633,4.0
CoxPH,0.631,5.5
CoxElastic,0.631,5.5
CoxLasso,0.630,7.0
CoxRidge,0.590,8.0


In [72]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

,IBS,rank
ExtraSurvivalTrees,0.218,1.0
Randomsurvivalforest,0.221,2.0
ComponentwiseGradientBoosting,0.224,3.0
GradientBoosting,0.230,4.0
CoxLasso,0.231,5.5
CoxElastic,0.231,5.5
CoxPH,0.232,7.0
CoxRidge,0.236,8.0


In [73]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

,C-index,rank
CoxRidge,0.642,1.0
Randomsurvivalforest,0.622,2.5
GradientBoosting,0.622,2.5
ComponentwiseGradientBoosting,0.601,4.0
ExtraSurvivalTrees,0.600,5.0
CoxLasso,0.599,6.5
CoxElastic,0.599,6.5
CoxPH,0.593,8.0


In [74]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs

,IBS,rank
ExtraSurvivalTrees,0.213,1.0
Randomsurvivalforest,0.214,2.5
ComponentwiseGradientBoosting,0.214,2.5
GradientBoosting,0.221,4.0
CoxRidge,0.229,5.0
CoxLasso,0.250,6.5
CoxElastic,0.250,6.5
CoxPH,0.253,8.0


In [75]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = 'path_to_your_folder/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']


dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d1/dfs/robust/plsr/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

# Modify the file names to match the desired format
modified_file_names = ['d1_dfs_robust_plsr_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file


In [76]:
from datetime import date
today = date.today()
print("Date: ", today)

Date:  2024-04-18
